# TaxoNova Mock Community Evaluation

We have classified the rep-seq.fasta and results are saved in final_taxonomy.tsv (observed taxonomies) and we have a trueish taxonomy txt file as a ground truth (expected taxonomies). Evaluate results with per sequence precision and outcome is detailed metrics for each comparison.

Metrics are: taxon detection rate, taxon accuracy rate, precision, recall and f-score.
Finally some preliminary plotting to visualize results.

In [1]:
## Import required libraries including parallel processing
import os
import sys
import random
import shutil
import glob
from joblib import Parallel, delayed
from os.path import join, basename, exists, expandvars, dirname, splitext, isfile
from pathlib import Path
from collections import defaultdict
import itertools
import numpy as np
import pandas as pd
from seaborn import xkcd_rgb as colors
import src.evaluate_results

# Parallel processing imports
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from functools import partial
import time
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner parallel output

print(f"Available CPU cores: {os.cpu_count()}")

Available CPU cores: 72


In [ ]:
## Set path vars - OPTIMIZED for parallel processing
n_cpus = os.cpu_count() - 1
n_jobs = min(n_cpus, 20)  # Optimized for parallel processing - use up to 8 cores
analysis_name = "mock-community"
project_dir = expandvars("/home/kardokhk/taxonova_bench/taxobench_eight")
conda_dir = join(project_dir, "conda")
scripts_dir = join(project_dir, f"scripts/src")
reference_database_dir = join(project_dir, f"benchmark/data/ref_dbs")
mock_dir = join(project_dir, f"benchmark/data", analysis_name)
results_dir = join(mock_dir, "results")

## Parameters
force = True # force run evaluations even if exists
taxa_level_ranges = range(2, 7) # 2 is class and 6 is species
seq_min_count = 1 # filter out ASV from biom is <1 hit
sample_weight = True # should the evaluations be done weighted by scale of abundance
per_seq_precision = True # evaluate Precision/Recall/Fscore on individual representative sequences

print(f"Using {n_jobs} parallel jobs out of {n_cpus} available CPUs")
print(f"Project directory: {project_dir}")
print(f"Mock directory: {mock_dir}")
print(f"Results directory: {results_dir}")

Using 30 parallel jobs out of 71 available CPUs
Project directory: /home/kardokhk/taxonova_bench/taxobench_five
Mock directory: /home/kardokhk/taxonova_bench/taxobench_five/benchmark/data/mock-community
Results directory: /home/kardokhk/taxonova_bench/taxobench_five/benchmark/data/mock-community/results


In [3]:
## Name of the mock communities
mock_names = ['gutmock', 'vaginalmock', 'mockrobiota-16S', 'mockrobiota-ITS']

## Amplicon regions
v_regions = ['V4', 'V3-V4', 'ITS1', 'ITS2', 'V1-V9', 'ITS12']

## Mock to v_region mapping
mock_v_regions = {
    'gutmock': ['V4', 'V3-V4', 'V1-V9'],
    'vaginalmock': ['V4', 'V3-V4', 'V1-V9'],
    'mockrobiota-16S': ['V4', 'V3-V4', 'V1-V9'],
    'mockrobiota-ITS': ['ITS1', 'ITS2', 'ITS12']
}

## Prepare combinations of methods and params to run for evaluations
method_parameters_combinations = {
    'taxonova': {'species_threshold': [0.5, 0.6, 0.7, 0.8],
                 'genus_threshold': [0.4, 0.5, 0.6],
                 'family_threshold': [0.3, 0.4],
                 'order_threshold': [0.2],
                 'class_threshold': [0.1],
                 'phylum_threshold': [0.1],
                 'domain_threshold': [0.1]
                },
    'vsearch': {'p-maxaccepts': [1, 10, 100],
                'p-perc-identity': [0.80, 0.90, 0.97, 0.99],
                'p-min-consensus': [0.51, 0.75, 0.99]
                },
    'blast': {'p-maxaccepts': [1, 10, 100],
                'p-perc-identity': [0.80, 0.90, 0.97, 0.99],
                'p-min-consensus': [0.51, 0.75, 0.99]
                },
    'naive-bayes': {'p-confidence': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
                },
    'idtaxa': {'p-confidence': [30, 40, 50, 60, 70, 80, 90]
                },
    'spingo': {'kmersize': [8],
               'bootstrap': [10, 20, 30, 40, 50, 60, 70, 80, 100],
               'subsample': [10]
                }                                       
}

## Taxonomy level mapping
taxonomy_level_map = {0:"Domain", 1:"Phylum", 2:"Class", 3:"Order", 4:"Family", 5:"Genus", 6:"Species"}

print(f"Mock communities: {mock_names}")
print(f"V-regions: {v_regions}")
print(f"Methods: {list(method_parameters_combinations.keys())}")
print(f"Taxonomic levels: {[taxonomy_level_map[i] for i in taxa_level_ranges]}")

# Supported taxonomy levels per classifier
classifier_supported_levels = {
    'taxonova': list(range(2, 7)),
    'vsearch': list(range(2, 7)),
    'blast': list(range(2, 7)),
    'naive-bayes': list(range(2, 7)),
    'idtaxa': list(range(2, 7)),
    'spingo': [5, 6]
}


Mock communities: ['gutmock', 'vaginalmock', 'mockrobiota-16S', 'mockrobiota-ITS']
V-regions: ['V4', 'V3-V4', 'ITS1', 'ITS2', 'V1-V9', 'ITS12']
Methods: ['taxonova', 'vsearch', 'blast', 'naive-bayes', 'idtaxa', 'spingo']
Taxonomic levels: ['Class', 'Order', 'Family', 'Genus', 'Species']


In [4]:
## Generate list of expected taxonomy filepaths
expected_taxonomy_filepaths = []
for v_region in v_regions:
    for mock_name in mock_names:
        # skip if region to mock mismatch
        if v_region not in mock_v_regions[mock_name]:
            continue
        
        table_fp = join(
            mock_dir,
            mock_name,
            v_region,
            'trueish-taxonomies.tsv'
        )

        expected_taxonomy_filepaths.append(
            (mock_name, v_region, table_fp)
        )

# Convert to dict for lookup
expected_taxonomy_filepaths_lookup = {(mock_name, v_region): file_path for mock_name, v_region, file_path in expected_taxonomy_filepaths}

print(f"Expected taxonomy files: {len(expected_taxonomy_filepaths)}")
for mock_name, v_region, file_path in expected_taxonomy_filepaths[:3]:  # Show first 3
    exists_status = "EXISTS" if exists(file_path) else "MISSING"
    print(f"  {mock_name}-{v_region}: {exists_status}")
if len(expected_taxonomy_filepaths) > 3:
    print(f"  ... and {len(expected_taxonomy_filepaths) - 3} more")

Expected taxonomy files: 12
  gutmock-V4: EXISTS
  vaginalmock-V4: EXISTS
  mockrobiota-16S-V4: EXISTS
  ... and 9 more


In [5]:
## Generate list of observed taxonomy filepaths
observed_taxonomy_filepaths = []
for v_region in v_regions:
    for mock_name in mock_names:
        # skip if region to mock mismatch
        if v_region not in mock_v_regions[mock_name]:
            continue
        
        for method_id, param_grid in method_parameters_combinations.items():
            param_names = list(param_grid.keys())
            param_values = [v if isinstance(v, list) else [v] for v in param_grid.values()]
            for combo in itertools.product(*param_values):
                param_id = ":".join(str(v) for v in combo)

                # Generate the observed table path
                table_fp = join(
                    mock_dir,
                    mock_name,
                    v_region,
                    'classify',
                    method_id,
                    param_id,
                    'taxonomy_final.tsv'
                )

                observed_taxonomy_filepaths.append(
                    (mock_name, v_region, method_id, param_id, table_fp)
                )

print(f"Total parameter combinations: {len(observed_taxonomy_filepaths)}")
print(f"Total evaluations (combinations × taxonomic levels): {sum(len(classifier_supported_levels.get(m, taxa_level_ranges)) for (_,_,m,_,_) in observed_taxonomy_filepaths)}")

# Count existing classification files
existing_files = sum(1 for _, _, _, _, fp in observed_taxonomy_filepaths if exists(fp))
print(f"Existing classification files: {existing_files}/{len(observed_taxonomy_filepaths)}")

# Show some examples
print("\nFirst few combinations:")
for i, (mock_name, v_region, method_id, param_id, table_fp) in enumerate(observed_taxonomy_filepaths[:5]):
    exists_status = "EXISTS" if exists(table_fp) else "MISSING"
    print(f"  {i+1}. {mock_name}-{v_region}-{method_id}-{param_id}: {exists_status}")

Total parameter combinations: 1476
Total evaluations (combinations × taxonomic levels): 7380
Existing classification files: 1476/1476

First few combinations:
  1. gutmock-V4-taxonova-0.5:0.4:0.3:0.2:0.1:0.1:0.1: EXISTS
  2. gutmock-V4-taxonova-0.5:0.4:0.4:0.2:0.1:0.1:0.1: EXISTS
  3. gutmock-V4-taxonova-0.5:0.5:0.3:0.2:0.1:0.1:0.1: EXISTS
  4. gutmock-V4-taxonova-0.5:0.5:0.4:0.2:0.1:0.1:0.1: EXISTS
  5. gutmock-V4-taxonova-0.5:0.6:0.3:0.2:0.1:0.1:0.1: EXISTS


In [6]:
## Generate expected_composition biom with metadata information
print("Generating expected composition BIOM tables...")

for v_region in v_regions:
    for mock_name in mock_names:
        # skip if region to mock mismatch
        if v_region not in mock_v_regions[mock_name]:
            continue
        
        ## prepare expected biom table with metadata information on taxonomic lineage
        exp_table_fp = join(mock_dir, mock_name, v_region, 'expected_composition.biom')
        env_abundance_fp = join(mock_dir, mock_name, v_region, 'env_abundance.tsv')
        
        if not exists(exp_table_fp) and exists(env_abundance_fp):
            try:
                src.evaluate_results.add_metadata_to_biom_table(
                    input_fp=env_abundance_fp,
                    taxonomy_map_fp=None,
                    biom_output_fp=exp_table_fp,
                    mode='expected'
                )
                print(f"  Generated: {mock_name}-{v_region}")
            except Exception as e:
                print(f"  Error generating {mock_name}-{v_region}: {e}")
        elif exists(exp_table_fp):
            print(f"  Exists: {mock_name}-{v_region}")
        else:
            print(f"  Missing source: {env_abundance_fp}")

print("Expected composition BIOM generation complete.")

Generating expected composition BIOM tables...
  Exists: gutmock-V4
  Exists: vaginalmock-V4
  Exists: mockrobiota-16S-V4
  Exists: gutmock-V3-V4
  Exists: vaginalmock-V3-V4
  Exists: mockrobiota-16S-V3-V4
  Exists: mockrobiota-ITS-ITS1
  Exists: mockrobiota-ITS-ITS2
  Exists: gutmock-V1-V9
  Exists: vaginalmock-V1-V9
  Exists: mockrobiota-16S-V1-V9
  Exists: mockrobiota-ITS-ITS12
Expected composition BIOM generation complete.


In [7]:
## PARALLEL BIOM TABLE GENERATION - Generate observed_composition biom with metadata information
print("Generating observed composition BIOM tables in parallel...")

def generate_observed_biom_single(args):
    """Generate observed composition biom table for a single classification result."""
    mock_id, region_id, method_id, param_id, obs_taxa_fp = args
    
    try:
        if not exists(obs_taxa_fp):
            return f"SKIP: {mock_id}-{region_id}-{method_id}-{param_id} (no taxonomy file)"
        
        obs_table_fp = join(os.path.dirname(obs_taxa_fp), 'observed_composition.biom')
        
        if exists(obs_table_fp):
            return f"EXISTS: {mock_id}-{region_id}-{method_id}-{param_id}"
        
        # Find source BIOM file
        source_biom_pattern = join(mock_dir, mock_id, region_id, 'source', '*', f'table_{mock_id}.biom')
        source_biom_files = glob.glob(source_biom_pattern)
        
        if not source_biom_files:
            return f"ERROR: {mock_id}-{region_id}-{method_id}-{param_id} (no source BIOM)"
        
        # Generate BIOM table
        src.evaluate_results.add_metadata_to_biom_table(
            input_fp=source_biom_files[0],
            taxonomy_map_fp=obs_taxa_fp,
            biom_output_fp=obs_table_fp,
            mode='observed'
        )
        
        return f"GENERATED: {mock_id}-{region_id}-{method_id}-{param_id}"
        
    except Exception as e:
        return f"ERROR: {mock_id}-{region_id}-{method_id}-{param_id} ({str(e)[:50]}...)"

# Prepare BIOM generation jobs
biom_jobs = [(mock_id, region_id, method_id, param_id, obs_taxa_fp) 
             for mock_id, region_id, method_id, param_id, obs_taxa_fp in observed_taxonomy_filepaths]

print(f"Processing {len(biom_jobs)} BIOM generation jobs with {n_jobs} workers...")
start_time = time.time()

# Use ThreadPoolExecutor for I/O-bound BIOM generation
with ThreadPoolExecutor(max_workers=n_jobs) as executor:
    biom_results = list(executor.map(generate_observed_biom_single, biom_jobs))

elapsed_time = time.time() - start_time
print(f"BIOM generation completed in {elapsed_time:.2f} seconds")

# Count results
generated = sum(1 for r in biom_results if r.startswith("GENERATED"))
existing = sum(1 for r in biom_results if r.startswith("EXISTS"))
errors = sum(1 for r in biom_results if r.startswith("ERROR"))
skipped = sum(1 for r in biom_results if r.startswith("SKIP"))

print(f"\nBIOM Generation Summary:")
print(f"  Generated: {generated}")
print(f"  Already existed: {existing}")
print(f"  Errors: {errors}")
print(f"  Skipped (no taxonomy): {skipped}")

if errors > 0:
    print("\nFirst few errors:")
    error_results = [r for r in biom_results if r.startswith("ERROR")][:3]
    for error in error_results:
        print(f"  {error}")

Generating observed composition BIOM tables in parallel...
Processing 1476 BIOM generation jobs with 30 workers...


BIOM generation completed in 5.12 seconds

BIOM Generation Summary:
  Generated: 234
  Already existed: 1242
  Errors: 0
  Skipped (no taxonomy): 0


In [8]:
## PARALLEL EVALUATION FUNCTION - Main optimization
def evaluate_single_combination_parallel(args):
    """
    Evaluate a single combination of (mock, region, method, param) across all taxonomic levels.
    This processes all taxa levels for one combination to minimize I/O overhead.
    """
    mock_id, region_id, method_id, param_id, obs_taxa_fp = args
    
    try:
        # Check if classification results exist
        if not exists(obs_taxa_fp):
            return None
        
        # Get paths
        exp_taxa_fp = expected_taxonomy_filepaths_lookup.get((mock_id, region_id))
        if not exp_taxa_fp or not exists(exp_taxa_fp):
            return None
        
        obs_table_fp = join(os.path.dirname(obs_taxa_fp), 'observed_composition.biom')
        feature_table = obs_table_fp  # Use same file for feature table
        exp_table_fp = join(mock_dir, mock_id, region_id, 'expected_composition.biom')
        
        # Check if required files exist
        if not all(exists(f) for f in [obs_table_fp, exp_table_fp]):
            return None
        
        # Process all taxonomic levels for this combination
        combination_results = []
        
        for taxa_level in classifier_supported_levels.get(method_id, taxa_level_ranges):
            try:
                # Generate unique output file for this specific combination and level
                outfile = join(os.path.dirname(obs_taxa_fp), f'evaluation_{taxa_level}.txt')
                
                # Skip if evaluation already exists and force=False
                if exists(outfile) and not force:
                    result = pd.read_csv(outfile, sep="\t", index_col=None)
                else:
                    result = src.evaluate_results.compute_results(
                        expected_table_fp=exp_table_fp, 
                        observed_table_fp=obs_table_fp, 
                        feature_table_fp=feature_table,
                        expected_taxa_fp=exp_taxa_fp, 
                        observed_taxa_fp=obs_taxa_fp,
                        taxonomy_level=taxa_level,
                        metadata=(mock_id, region_id, method_id, param_id, taxonomy_level_map[taxa_level]),
                        out_file=outfile,
                        min_count=seq_min_count,
                        sample_weight=sample_weight,
                        per_seq_precision=per_seq_precision,
                        md_key='taxonomy',
                        taxa_to_keep=None,
                        exclude=None
                    )
                
                if result is not None:
                    combination_results.append(result)
                
            except Exception as e:
                print(f"Error processing {mock_id}-{region_id}-{method_id}-{param_id} at level {taxa_level}: {e}")
                continue
        
        return combination_results
        
    except Exception as e:
        print(f"Error processing combination {mock_id}-{region_id}-{method_id}-{param_id}: {e}")
        return None

print("Parallel evaluation function defined.")

Parallel evaluation function defined.


In [9]:
## MAIN PARALLEL EVALUATION - Replace the serial evaluation loop
print("=" * 60)
print("STARTING PARALLEL EVALUATION")
print("=" * 60)

# Filter for combinations that have both taxonomy and BIOM files
valid_combinations = []
for combo in observed_taxonomy_filepaths:
    mock_id, region_id, method_id, param_id, obs_taxa_fp = combo
    obs_table_fp = join(os.path.dirname(obs_taxa_fp), 'observed_composition.biom')
    
    if exists(obs_taxa_fp) and exists(obs_table_fp):
        valid_combinations.append(combo)

print(f"Valid combinations to evaluate: {len(valid_combinations)}/{len(observed_taxonomy_filepaths)}")
print(f"Total individual evaluations: {sum(len(classifier_supported_levels.get(m, taxa_level_ranges)) for (_,_,m,_,_) in valid_combinations)}")
print(f"Using {n_jobs} parallel workers")
print(f"Expected speedup: ~{min(n_jobs, len(valid_combinations))}x")

if len(valid_combinations) == 0:
    print("\nWARNING: No valid combinations found to evaluate!")
    print("This might be because:")
    print("1. Classification hasn't been run yet")
    print("2. BIOM table generation failed")
    print("3. File paths are incorrect")
    final_result = pd.DataFrame()
else:
    print(f"\nStarting parallel evaluation...")
    start_time = time.time()
    
    # Run parallel evaluation using ProcessPoolExecutor for CPU-bound tasks
    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        parallel_results = list(executor.map(evaluate_single_combination_parallel, valid_combinations))
    
    # Collect and combine results
    all_results = []
    for result_list in parallel_results:
        if result_list is not None:
            for result in result_list:
                if result is not None:
                    all_results.append(result)
    
    elapsed_time = time.time() - start_time
    
    print(f"\nParallel evaluation completed!")
    print(f"Time elapsed: {elapsed_time:.2f} seconds")
    print(f"Results collected: {len(all_results)}")
    
    if all_results:
        # Combine all into one final dataframe
        final_result = pd.concat(all_results, ignore_index=True)
        print(f"Final result shape: {final_result.shape}")
        
        # Calculate performance stats
        total_evaluations = len(all_results)
        evaluations_per_second = total_evaluations / elapsed_time if elapsed_time > 0 else 0
        
        print(f"\nPerformance Summary:")
        print(f"  Total evaluations: {total_evaluations}")
        print(f"  Evaluations per second: {evaluations_per_second:.2f}")
        print(f"  Average time per evaluation: {elapsed_time/total_evaluations:.3f} seconds")
        
    else:
        print("No results generated")
        final_result = pd.DataFrame()

print("\nParallel evaluation phase complete.")

STARTING PARALLEL EVALUATION
Valid combinations to evaluate: 1476/1476
Total individual evaluations: 7380
Using 30 parallel workers
Expected speedup: ~30x

Starting parallel evaluation...



Parallel evaluation completed!
Time elapsed: 106.24 seconds
Results collected: 7380


Final result shape: (221400, 20)

Performance Summary:
  Total evaluations: 7380
  Evaluations per second: 69.47
  Average time per evaluation: 0.014 seconds

Parallel evaluation phase complete.


In [10]:
## Display results summary
if not final_result.empty:
    print("\nEvaluation Results Summary:")
    print(f"Total rows: {len(final_result)}")
    print(f"Columns: {list(final_result.columns)}")
    
    # Show breakdown by method and mock
    if 'method' in final_result.columns:
        print("\nResults by method:")
        method_counts = final_result['method'].value_counts()
        for method, count in method_counts.items():
            print(f"  {method}: {count} evaluations")
    
    if 'mock_name' in final_result.columns:
        print("\nResults by mock community:")
        mock_counts = final_result['mock_name'].value_counts()
        for mock, count in mock_counts.items():
            print(f"  {mock}: {count} evaluations")
    
    # Show first few rows
    print("\nFirst few results:")
    display_cols = ['mock_name', 'v_region', 'method', 'taxa_level', 'Precision', 'Recall', 'F-measure']
    available_cols = [col for col in display_cols if col in final_result.columns]
    print(final_result[available_cols].head())
    
else:
    print("\nNo results to display - final_result is empty")
    final_result = pd.DataFrame()  # Ensure it's defined


Evaluation Results Summary:
Total rows: 221400
Columns: ['mock_name', 'v_region', 'method', 'param', 'taxa_level', 'sample_id', 'n_exp_taxa', 'n_obs_taxa', 'tp', 'fp', 'fn', 'Taxon Accuracy Rate', 'Taxon Detection Rate', 'n_exp_seq', 'n_obs_seq', 'Precision', 'Recall', 'F-measure', 'per_seq_ids', 'per_seq_matches']

Results by method:
  vsearch: 64800 evaluations
  blast: 64800 evaluations
  taxonova: 43200 evaluations
  naive-bayes: 19800 evaluations
  spingo: 16200 evaluations
  idtaxa: 12600 evaluations

Results by mock community:
  gutmock: 55350 evaluations
  vaginalmock: 55350 evaluations
  mockrobiota-16S: 55350 evaluations
  mockrobiota-ITS: 55350 evaluations

First few results:
  mock_name v_region    method taxa_level  Precision  Recall  F-measure
0   gutmock       V4  taxonova      Class        1.0     1.0        1.0
1   gutmock       V4  taxonova      Class        1.0     1.0        1.0
2   gutmock       V4  taxonova      Class        1.0     1.0        1.0
3   gutmock    

In [11]:
## Export final_result table as TSV file
if not final_result.empty:
    # Create results directory if it doesn't exist
    if not exists(results_dir):
        os.makedirs(results_dir)
        print(f"Created results directory: {results_dir}")
    
    # Save the final_result table
    tsv_output_path = join(results_dir, "final_result_evaluation_parallel.tsv")
    final_result.to_csv(tsv_output_path, sep='\t', index=False)
    
    print(f"\nFinal result table exported to: {tsv_output_path}")
    print(f"File size: {os.path.getsize(tsv_output_path) / 1024:.1f} KB")
    
    # Also save a backup with timestamp
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = join(results_dir, f"final_result_evaluation_parallel_{timestamp}.tsv")
    final_result.to_csv(backup_path, sep='\t', index=False)
    print(f"Backup saved to: {backup_path}")
    
else:
    print("\nNo results to export - final_result is empty")
    tsv_output_path = None


Final result table exported to: /home/kardokhk/taxonova_bench/taxobench_five/benchmark/data/mock-community/results/final_result_evaluation_parallel.tsv
File size: 551336.0 KB


Backup saved to: /home/kardokhk/taxonova_bench/taxobench_five/benchmark/data/mock-community/results/final_result_evaluation_parallel_20250613_234444.tsv


In [12]:
## Reload evaluate_results module for plotting
from importlib import reload
reload(src.evaluate_results)

print("evaluate_results module reloaded for plotting")

evaluate_results module reloaded for plotting


In [13]:
## Quick Plot - only if we have results
if not final_result.empty:
    print("Generating evaluation plots...")
    
    ## Plot parameters
    only_plot_taxonova = False
    plot_by_param = False
    plot_by_region = True  # Add new parameter for plotting by region
    if only_plot_taxonova: 
        plot_by_param = True

    plot_by_mock = True
    if plot_by_mock:
        line_by = "mock_name"
    else:
        line_by = None

    ## Output directory
    out_dir = results_dir
    if not exists(out_dir):
        os.makedirs(out_dir)

    ## Reset index
    mock_results = final_result.reset_index(drop=True)

    ## Color palette
    color_palette = {
        'rdp': colors['baby shit green'], 
        'sortmerna': colors['macaroni and cheese'],
        'uclust': 'coral', 
        'blast': colors['electric purple'], 
        'naive-bayes': 'dodgerblue',
        'naive-bayes-bespoke': 'blue', 
        'vsearch': 'firebrick', 
        'taxonova': 'black', 
        'idtaxa': 'green', 
        'spingo': 'orange'
    }

    ## Line styles
    line_dict = {
        "mockrobiota-16S": "", # solid
        "mockrobiota-ITS": (1, 1), # dotted
        "gutmock": (2, 2), # dashed
        "vaginalmock": (3, 2, 1, 2), # dash-dot pattern
        "skinmock": (5, 2), # long dash
    }

    ## Evaluation metrics to plot
    y_vars = ["Precision", "Recall", "F-measure", "Taxon Accuracy Rate", "Taxon Detection Rate"]
    
    # Check which columns are actually available
    available_y_vars = [var for var in y_vars if var in mock_results.columns]
    if len(available_y_vars) != len(y_vars):
        print(f"Warning: Some metrics not available. Using: {available_y_vars}")
        y_vars = available_y_vars

    ## Only plot results for taxonova if requested
    if only_plot_taxonova and 'method' in mock_results.columns:
        mock_results = mock_results[mock_results['method'] == "taxonova"]
        print(f"Filtered to taxonova only: {len(mock_results)} rows")

    print(f"Plotting {len(mock_results)} results with metrics: {y_vars}")
    
    try:
        ## Plot results
        if not plot_by_param and not plot_by_region:
            # plot results from all params and regions combined
            print("Generating combined plot...")
            src.evaluate_results.plot_from_data_frame(
                df=mock_results,
                outdir=out_dir,
                x_axis="taxa_level",
                y_vars=y_vars,
                color_by="method",
                color_palette=color_palette,
                line_by=line_by
            )

        elif plot_by_region and 'v_region' in mock_results.columns:
            # plot all regions separately with organized directories
            unique_regions = mock_results['v_region'].unique()
            print(f"Plotting by region: {list(unique_regions)}")
            
            for region in unique_regions:
                # Create directory for each region
                region_dir = join(out_dir, f"by_region/{region}")
                if not exists(region_dir):
                    os.makedirs(region_dir)
                
                region_subset = mock_results[mock_results['v_region'] == region]
                print(f"  Plotting {region}: {len(region_subset)} results")
                
                if plot_by_param and 'param' in region_subset.columns:
                    # plot by param within each region
                    unique_params = region_subset['param'].unique()
                    
                    for param in unique_params:
                        param_subset = region_subset[region_subset['param'] == param]
                        
                        src.evaluate_results.plot_from_data_frame(
                            df=param_subset,
                            outdir=region_dir,
                            x_axis="taxa_level",
                            y_vars=y_vars,
                            color_by="method",
                            color_palette=color_palette,
                            line_by=line_by,
                            out_file=f"{region}_{param}_eval_results.pdf"
                        )
                else:
                    # plot all params combined for each region
                    src.evaluate_results.plot_from_data_frame(
                        df=region_subset,
                        outdir=region_dir,
                        x_axis="taxa_level",
                        y_vars=y_vars,
                        color_by="method",
                        color_palette=color_palette,
                        line_by=line_by,
                        out_file=f"{region}_eval_results.pdf"
                    )

        elif plot_by_param and 'param' in mock_results.columns:
            # plot all params separately (original functionality)
            unique_params = mock_results['param'].unique()
            print(f"Plotting by parameter: {len(unique_params)} unique parameters")

            for param in unique_params:
                subset_df = mock_results[mock_results['param'] == param]

                src.evaluate_results.plot_from_data_frame(
                    df=subset_df,
                    outdir=out_dir,
                    x_axis="taxa_level",
                    y_vars=y_vars,
                    color_by="method",
                    color_palette=color_palette,
                    line_by=line_by,
                    out_file=f"{param}_eval_results.pdf"
                )
        
        print(f"\nPlotting completed. Plots saved to: {out_dir}")
        
    except Exception as e:
        print(f"\nError during plotting: {e}")
        print("Plotting failed, but evaluation results are still available.")

else:
    print("\nSkipping plots - no results available")

Generating evaluation plots...
Plotting 221400 results with metrics: ['Precision', 'Recall', 'F-measure', 'Taxon Accuracy Rate', 'Taxon Detection Rate']
Plotting by region: ['V4', 'V3-V4', 'ITS1', 'ITS2', 'V1-V9', 'ITS12']


  Plotting V4: 55350 results


  Plotting V3-V4: 55350 results


  Plotting ITS1: 18450 results


  Plotting ITS2: 18450 results


  Plotting V1-V9: 55350 results


  Plotting ITS12: 18450 results



Plotting completed. Plots saved to: /home/kardokhk/taxonova_bench/taxobench_five/benchmark/data/mock-community/results


In [ ]:
## Species-Level Bar Chart - V-regions on x-axis, methods as grouped bars
if not final_result.empty and 'species_data' in locals() and not species_data.empty:
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.backends.backend_pdf import PdfPages
    import numpy as np
    
    print("Generating species-level bar charts by V-region...")
    
    # Enhanced color palette with more vibrant colors
    enhanced_color_palette = {
        'rdp': '#2E8B57',          # Sea Green
        'sortmerna': '#FF6347',    # Tomato
        'uclust': '#FF7F50',       # Coral
        'blast': '#8A2BE2',        # Blue Violet
        'naive-bayes': '#1E90FF',  # Dodger Blue
        'naive-bayes-bespoke': '#0000CD',  # Medium Blue
        'vsearch': '#DC143C',      # Crimson
        'taxonova': '#FF69B4',     # Black
        'idtaxa': '#228B22',       # Forest Green
        'spingo': '#FF8C00'        # Dark Orange
    }
    
    # Enhanced hatching patterns for better distinction
    enhanced_hatch_patterns = {
        'rdp': '|||',           # No hatch (solid)
        'sortmerna': '///',  # Diagonal lines
        'uclust': '\\\\\\',     # Dots
        'blast': 'xxx',      # Crosses
        'naive-bayes': '+++', # Plus signs
        'naive-bayes-bespoke': '---',  # Horizontal lines
        'vsearch': '...',    # Vertical lines
        'taxonova': '', # Reverse diagonal
        'idtaxa': '***',     # Stars
        'spingo': 'ooo'      # Circles
    }
    
    # Calculate mean and standard deviation for bar chart and error bars
    bar_stats = species_data.groupby(['v_region', 'method'])[available_metrics].agg(['mean', 'std']).reset_index()
    
    # Set up the plotting style
    sns.set_style("whitegrid")
    sns.set_context("notebook", font_scale=1.2)
    
    # Create figure with subplots for each metric
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 14))
    axes = axes.flatten()
    
    # Get unique methods and regions for consistent ordering
    unique_methods = sorted(species_data['method'].unique())
    unique_regions = sorted(species_data['v_region'].unique())
    
    # Plot only the first 5 metrics (save 6th subplot for legend)
    metrics_to_plot = available_metrics[:5]
    
    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]
        
        # Prepare data for grouped bar chart
        region_positions = np.arange(len(unique_regions))
        bar_width = 0.8 / len(unique_methods)
        
        # Create bars for each method
        for j, method in enumerate(unique_methods):
            method_means = []
            method_stds = []
            
            for region in unique_regions:
                # Get mean and std for this method-region combination
                subset = bar_stats[(bar_stats['method'] == method) & (bar_stats['v_region'] == region)]
                if not subset.empty:
                    mean_val = subset[(metric, 'mean')].iloc[0]
                    std_val = subset[(metric, 'std')].iloc[0]
                    method_means.append(mean_val if not np.isnan(mean_val) else 0)
                    method_stds.append(std_val if not np.isnan(std_val) else 0)
                else:
                    method_means.append(0)
                    method_stds.append(0)
            
            # Calculate bar positions
            bar_positions = region_positions + (j - len(unique_methods)/2 + 0.5) * bar_width
            
            # Create bars with color and hatching
            bars = ax.bar(
                bar_positions, 
                method_means,
                bar_width,
                yerr=method_stds,  # Add error bars for variability
                label=method,
                color=enhanced_color_palette.get(method, 'gray'),
                hatch=enhanced_hatch_patterns.get(method, ''),
                edgecolor='black',
                linewidth=0.8,
                alpha=0.8,
                capsize=3,  # Error bar cap size
                ecolor='black',  # Error bar color
                error_kw={'linewidth': 1.0}  # Error bar line width via error_kw
            )
            
            # Add rotated value labels on top of error bars
            for bar, mean_val, std_val in zip(bars, method_means, method_stds):
                if mean_val > 0.05:  # Only show labels for meaningful values
                    # Position label above error bar
                    label_height = bar.get_height() + std_val + 0.02
                    ax.text(bar.get_x() + bar.get_width()/2, label_height,
                           f'{mean_val:.2f}', 
                           ha='center', va='bottom', 
                           fontsize=8, 
                           rotation=90,  # Rotate 90 degrees
                           fontweight='normal',
                           color='black')
        
        # Customize the plot
        ax.set_title(f'{metric}', fontsize=14, fontweight='bold')
        ax.set_xlabel('V-Region', fontsize=12)
        ax.set_ylabel(f'Mean {metric}', fontsize=12)
        ax.tick_params(labelsize=10)
        
        # Set x-axis
        ax.set_xticks(region_positions)
        ax.set_xticklabels(unique_regions, rotation=45, ha='right')
        
        # Set y-axis limits with extra space for rotated labels
        ax.set_ylim(0, 1.2)  # Increased upper limit for rotated labels
        ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
        
        # Add grid for better readability
        ax.grid(True, alpha=0.3, axis='y')
        
        # Remove any legends from individual subplots
        if ax.get_legend():
            ax.get_legend().remove()
    
    # Use the 6th subplot (index 5) for the legend
    legend_ax = axes[5]
    legend_ax.axis('off')  # Turn off axis for legend subplot
    
    # Create enhanced legend elements with both color and hatch pattern information
    legend_elements = []
    for method in unique_methods:
        if method in enhanced_color_palette:
            from matplotlib.patches import Patch
            legend_elements.append(
                Patch(facecolor=enhanced_color_palette[method], 
                      hatch=enhanced_hatch_patterns.get(method, ''),
                      edgecolor='black',
                      linewidth=0.8,
                      label=method)
            )
    
    # Place legend in the 6th subplot with vertical arrangement (one classifier per line)
    legend_ax.legend(handles=legend_elements, 
                     labels=[elem.get_label() for elem in legend_elements],
                     title='',
                     loc='center', 
                     fontsize=12, 
                     title_fontsize=14,
                     frameon=False, 
                     fancybox=False, 
                     shadow=False,
                     ncol=1)  # Force single column for vertical listing
    
    plt.tight_layout()
    
    # Save the bar chart
    bar_chart_file = join(species_plot_dir, "species_level_barchart_by_region.pdf")
    with PdfPages(bar_chart_file) as pdf:
        pdf.savefig(fig, bbox_inches='tight', dpi=300)
    
    plt.close()
    
    print(f"Species-level bar chart saved to: {bar_chart_file}")
    
    # Create a comparison summary table
    print(f"\nBar Chart Summary (Mean ± SD at species level):")
    
    # Create a more detailed summary with both mean and std
    for metric in available_metrics:
        print(f"\n{metric}:")
        for region in unique_regions:
            print(f"  {region}:")
            for method in unique_methods:
                subset = bar_stats[(bar_stats['method'] == method) & (bar_stats['v_region'] == region)]
                if not subset.empty:
                    mean_val = subset[(metric, 'mean')].iloc[0]
                    std_val = subset[(metric, 'std')].iloc[0]
                    if not np.isnan(mean_val):
                        print(f"    {method}: {mean_val:.3f} ± {std_val:.3f}")
    
    # Find best performing method per region for F-measure
    if 'F-measure' in available_metrics:
        print(f"\nBest performing method by V-region (F-measure, mean ± SD):")
        for region in unique_regions:
            best_score = 0
            best_method = None
            best_std = 0
            
            for method in unique_methods:
                subset = bar_stats[(bar_stats['method'] == method) & (bar_stats['v_region'] == region)]
                if not subset.empty:
                    mean_val = subset[('F-measure', 'mean')].iloc[0]
                    std_val = subset[('F-measure', 'std')].iloc[0]
                    if not np.isnan(mean_val) and mean_val > best_score:
                        best_score = mean_val
                        best_method = method
                        best_std = std_val
            
            if best_method:
                print(f"  {region}: {best_method} ({best_score:.3f} ± {best_std:.3f})")
    
else:
    print("Skipping bar chart - no species data available or previous box plot not generated")

In [14]:
## Final Summary
print("=" * 60)
print("EVALUATION COMPLETE - PERFORMANCE SUMMARY")
print("=" * 60)

print(f"\nConfiguration:")
print(f"  Parallel workers used: {n_jobs}")
print(f"  Total CPU cores available: {os.cpu_count()}")
print(f"  Mock communities: {len(mock_names)}")
print(f"  V-regions: {len(v_regions)}")
print(f"  Methods: {len(method_parameters_combinations)}")
print(f"  Taxonomic levels: {sum(len(v) for v in classifier_supported_levels.values())/len(classifier_supported_levels)}")

print(f"\nResults:")
if not final_result.empty:
    print(f"  Total evaluations completed: {len(final_result)}")
    print(f"  Results file: {tsv_output_path if 'tsv_output_path' in locals() and tsv_output_path else 'Not saved'}")
    print(f"  Plots directory: {out_dir if 'out_dir' in locals() else 'Not generated'}")
    
    # Show some key statistics
    if 'Precision' in final_result.columns:
        avg_precision = final_result['Precision'].mean()
        print(f"  Average Precision: {avg_precision:.3f}")
    
    if 'Recall' in final_result.columns:
        avg_recall = final_result['Recall'].mean()
        print(f"  Average Recall: {avg_recall:.3f}")
    
    if 'F-measure' in final_result.columns:
        avg_fscore = final_result['F-measure'].mean()
        print(f"  Average F-measure: {avg_fscore:.3f}")
else:
    print(f"  No results generated - check input files and paths")

print(f"\nOptimizations Applied:")
print(f"  ✅ Parallel processing of parameter combinations")
print(f"  ✅ Parallel BIOM table generation")
print(f"  ✅ Efficient error handling")
print(f"  ✅ Memory-optimized processing")
print(f"  ✅ Progress reporting")

print(f"\n🚀 Evaluation framework optimized for {n_jobs}x parallel speedup!")
print(f"\nTo run this notebook:")
print(f"jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=60000 --log-level=INFO scripts/evaluate_mock_community_optimized.ipynb")

print("\n" + "=" * 60)

EVALUATION COMPLETE - PERFORMANCE SUMMARY

Configuration:
  Parallel workers used: 30
  Total CPU cores available: 72
  Mock communities: 4
  V-regions: 6
  Methods: 6
  Taxonomic levels: 5

Results:
  Total evaluations completed: 221400
  Results file: /home/kardokhk/taxonova_bench/taxobench_five/benchmark/data/mock-community/results/final_result_evaluation_parallel.tsv
  Plots directory: /home/kardokhk/taxonova_bench/taxobench_five/benchmark/data/mock-community/results
  Average Precision: 0.811
  Average Recall: 0.700
  Average F-measure: 0.729

Optimizations Applied:
  ✅ Parallel processing of parameter combinations
  ✅ Parallel BIOM table generation
  ✅ Efficient error handling
  ✅ Memory-optimized processing
  ✅ Progress reporting

🚀 Evaluation framework optimized for 30x parallel speedup!

To run this notebook:
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=60000 --log-level=INFO scripts/evaluate_mock_community_optimized.ipynb

